# Évaluer le RAG de la presse de 1914

Le RAG du chapitre 2 répond aux questions sur les 64 598 passages de la presse de 1914 : recherche hybride (BM25 et recherche dense fusionnés par RRF), reclassement par un cross-encodeur, puis une réponse sourcée, ou un refus, écrite par un LLM servi par Ollama. Il a été réglé sur 20 questions. Ce TP l'évalue comme un système qu'on met en service : sur un jeu de test plus large, une mesure après l'autre, et d'une version à la suivante.

1. Générer des questions depuis le corpus
2. Filtrer, valider et assembler un jeu de test mixte
3. Mesurer la recherche : dense (avant), hybride, hybride avec reclassement (après)
4. Mesurer la réponse avec Ragas : fidélité, exactitude, refus, citations
5. Contrôler le juge sur des réponses annotées
6. Consigner chaque évaluation et comparer question par question

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Le RAG y répond avec Qwen3-4B, et un modèle plus gros, d'une autre famille, le juge : Ministral 3 8B, de Mistral AI. Sans GPU, il tourne quand même, en quarante minutes environ, en plus petit : Qwen3-1.7B y répond et juge, le reclassement n'est mesuré que sur 4 questions, et Ragas ne note que quelques réponses.

## Données et modèles

Les passages et leurs *embeddings* (512 dimensions), les 20 questions du chapitre 2, des questions générées puis validées (exercice 2) et des réponses annotées (exercices 4 et 5). Ollama s'installe et télécharge les modèles en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/questions.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/synthetic-questions.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/annotated-answers.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bm25s ragas

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# Le LLM du RAG : sur GPU, Qwen3-4B ; sur CPU, Qwen3-1.7B, deux fois plus rapide : un
# modèle « hybride », qui réfléchit avant de répondre, sauf avec reasoning_effort="none".
MODELE = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# Le juge : sur GPU, Ministral 3 8B (Mistral AI), plus gros et d'une autre famille ; sur
# CPU, le même Qwen3-1.7B (voir l'exercice 4).
MODELE_JUGE = "ministral-3:8b" if GPU else "qwen3:1.7b"
print("GPU" if GPU else "Sans GPU", ":", MODELE, "répond,", MODELE_JUGE, "juge")
# Sur GPU, 4 demandes à la fois ; sur CPU, une : le début commun des prompts (les
# consignes) n'est alors calculé qu'une fois. 8 192 jetons de contexte par demande : le
# prompt de fidélité de Ragas, avec ses exemples et les 5 passages, dépasse les 4 096 par
# défaut, et Ollama couperait le début du prompt sans rien dire.
paralleles = "4" if GPU else "1"
env = {
  **os.environ,
  "OLLAMA_NUM_PARALLEL": paralleles,
  "OLLAMA_CONTEXT_LENGTH": "8192",
  "OLLAMA_KEEP_ALIVE": "-1",
}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE} && ollama pull {MODELE_JUGE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

## Le RAG à évaluer

Le RAG du chapitre 2, donné : `classer(question, configuration)` renvoie le classement des passages (leurs positions dans `passages`) selon l'une des trois configurations de la recherche, `"dense"` (la recherche dense seule), `"hybride"` (BM25 et dense, 100 candidats chacun, fusionnés par RRF) et `"hybride+reclassement"` (les `A_RECLASSER` premiers, 30, reclassés par le cross-encodeur). Chaque classement est calculé une fois, puis gardé en mémoire. `rechercher(question, k, configuration)` en garde les `k` premiers, et `repondre(question, configuration)` y ajoute la réponse du LLM, comme à l'exercice 3 du chapitre 2. Lancez la cellule (deux à trois minutes, durée indicative : elle attend aussi les LLM).

In [ ]:
# bm25s se sert de JAX s'il est installé (c'est le cas sur Colab), et JAX réserve d'office
# 75 % de la mémoire du GPU : on le cantonne au CPU, le GPU est pour les modèles.
os.environ["JAX_PLATFORMS"] = "cpu"

import asyncio
import hashlib
import json
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from functools import cache

import bm25s
import numpy as np
import pandas as pd
from openai import AsyncOpenAI, LengthFinishReasonError, OpenAI
from pydantic import BaseModel, ValidationError
from sentence_transformers import CrossEncoder, SentenceTransformer

telechargement.wait()  # les modèles sont téléchargés
# Un long délai : sans GPU, une demande peut attendre son tour plusieurs minutes.
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=3600)
# Le LLM d'abord : un premier appel le charge (sur le GPU s'il y en a un), avant les
# autres modèles.
client.chat.completions.create(
  model=MODELE, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1
)
print(MODELE, "prêt")

passages = pd.read_parquet("passages.parquet")
par_id = passages.set_index("id")
vecteurs = np.load("embeddings-512.npy").astype(np.float32)
questions = pd.read_json("questions.jsonl", lines=True)
# Une question sans réponse a `answer` à null : pandas en fait un NaN, que `or` prend
# pour une valeur. On le remplace une fois pour toutes par None.
questions["answer"] = questions.answer.astype(object).where(
  questions.answer.notna(), None
)

DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
ENCODEUR, RECLASSEUR = "Qwen/Qwen3-Embedding-0.6B", "BAAI/bge-reranker-v2-m3"
encodeur = SentenceTransformer(
  ENCODEUR, device=DEVICE, truncate_dim=512, model_kwargs={"dtype": DTYPE}
)
reclasseur = CrossEncoder(
  RECLASSEUR, device=DEVICE, max_length=512, model_kwargs={"dtype": DTYPE}
)
A_RECLASSER = 30
index_mots = bm25s.BM25()
index_mots.index(
  bm25s.tokenize(passages.text.tolist(), show_progress=False), show_progress=False
)


def rrf(*classements, k: int = 60) -> list[int]:
  scores = defaultdict(float)
  for classement in classements:
    for rang, doc in enumerate(classement, start=1):
      scores[doc] += 1 / (k + rang)
  return sorted(scores, key=scores.get, reverse=True)


@cache
def classer_dense(question: str) -> list[int]:
  requete = encodeur.encode_query(question).astype(np.float32)
  requete /= np.linalg.norm(requete)
  return np.argsort(-(vecteurs @ requete))[:100].tolist()


@cache
def classer(question: str, configuration: str = "hybride+reclassement") -> list[int]:
  dense = classer_dense(question)
  if configuration == "dense":
    return dense
  lexicale = index_mots.retrieve(
    bm25s.tokenize([question], show_progress=False), k=100, show_progress=False
  )[0][0]
  candidats = rrf(lexicale.tolist(), dense)
  if configuration == "hybride":
    return candidats
  premiers = candidats[:A_RECLASSER]
  scores = reclasseur.predict(
    [(question, passages.text[i]) for i in premiers], batch_size=32
  )
  return [premiers[i] for i in np.argsort(-scores)]


def rechercher(
  question: str, k: int = 5, configuration: str = "hybride+reclassement"
) -> list[int]:
  return classer(question, configuration)[:k]


class Reponse(BaseModel):
  trouve: bool
  reponse: str
  citations: list[int]


INSTRUCTIONS = """Tu réponds aux questions sur la presse française de 1914, uniquement à partir
des passages fournis, numérotés. Réponds en français, en une ou deux phrases, et mets dans
citations les numéros des passages qui justifient ta réponse (pas dans la réponse elle-même).
Si les passages ne contiennent pas la réponse, mets trouve à false et n'invente rien."""


def repondre(question: str, configuration: str = "hybride+reclassement") -> dict:
  extraits = rechercher(question, 5, configuration)
  contexte = "\n\n".join(
    f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"
    for n, i in enumerate(extraits, start=1)
  )
  servis = passages.id[extraits].tolist()
  try:
    retour = client.chat.completions.parse(
      model=MODELE,
      messages=[
        {"role": "system", "content": INSTRUCTIONS},
        {"role": "user", "content": f"Passages :\n{contexte}\n\nQuestion : {question}"},
      ],
      response_format=Reponse,
      temperature=0,
      reasoning_effort="none",
      max_tokens=300,
    )
  except (LengthFinishReasonError, ValidationError):
    # Un petit modèle boucle parfois dans son JSON (coupé par max_tokens), ou en écrit un
    # illisible : la réponse compte pour un refus.
    return {"trouve": False, "reponse": "", "servis": servis, "sources": []}
  resultat = retour.choices[0].message.parsed
  sources = [servis[n - 1] for n in resultat.citations if 1 <= n <= len(servis)]
  return {
    "trouve": resultat.trouve,
    "reponse": resultat.reponse,
    "servis": servis,
    "sources": sources,
  }


for i in rechercher("Où Jaurès a-t-il été assassiné ?", 3, "hybride"):
  print(passages.id[i], passages.text[i][:200])

## 1. Générer des questions depuis le corpus

Écrire des questions à la main prend du temps, et on n'en écrit jamais assez. Un LLM peut en écrire : on lui donne un passage tiré au hasard, il écrit une question et sa réponse, et le passage de référence est connu par construction. Écrivez `generer_question(i)` : pour le passage à la position `i` de `passages`, elle demande au modèle un objet `QuestionGeneree` (ci-dessous) et renvoie un dictionnaire : l'identifiant du passage (`passage_id`) et les trois champs. Sa consigne, `CONSIGNES_GENERATION`, demande une question qu'un lecteur d'aujourd'hui pourrait poser, compréhensible sans le passage, sur un fait précis que le passage donne, avec sa réponse courte, et `exploitable` à `false` quand le passage ne donne aucun fait précis ; deux bonnes et deux mauvaises questions en exemple montrent ce qu'on attend.

Lancez-la sur les passages de `tirage` (8 ; sans GPU, 3) et lisez les questions : se comprennent-elles sans le passage ? La réponse est-elle juste ?

Un petit modèle s'acquitte mal de cette tâche : Qwen3-1.7B, le modèle sans GPU, donne surtout des questions inutilisables (« dans le passage », un fait mal lu, le bruit de l'OCR recopié). D'où, pour le jeu de l'exercice 2, un générateur bien plus fort, et une relecture quand même : ses 60 questions ont été générées par Claude (Sonnet, d'Anthropic) avec cette consigne, un appel par passage, sur le même tirage étendu à 60 passages, puis validées une à une. Dans un projet, cette relecture revient à un expert métier ; ici, un autre LLM (Claude, dans un contexte séparé) en a tenu le rôle, et a aussi annoté les réponses de l'exercice 5 : ses verdicts se discutent comme ceux d'un collègue.

**Documentation utile**

- [`BaseModel.model_dump`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)

In [ ]:
# Les champs dans cet ordre : le modèle écrit la question et la réponse avant de juger
# l'extrait exploitable (en tête, il déciderait avant de l'avoir lu).
class QuestionGeneree(BaseModel):
  question: str
  reponse: str
  exploitable: bool


# Le tirage : des passages peu bruités, deux tiers de L'Humanité, un tiers du Figaro, de
# toute l'année. Sur 60 passages, ce même tirage a donné les questions de l'exercice 2 :
# ses premiers passages sont ceux-ci. Sans GPU, une minute environ par question (durée
# indicative).
N_TIRES = 8 if GPU else 3
propres = passages[passages.noise <= 0.05]
tirage = pd.concat(
  [
    propres[propres.newspaper == "L'Humanité"].sample(
      N_TIRES * 2 // 3, random_state=1914
    ),
    propres[propres.newspaper == "Le Figaro"].sample(
      N_TIRES - N_TIRES * 2 // 3, random_state=1914
    ),
  ]
).sort_values("date")

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `client.chat.completions.parse(model=MODELE, messages=[...], response_format=QuestionGeneree, temperature=0, reasoning_effort="none", max_tokens=200)`, comme `repondre` : la consigne en message `system`, le passage en message `user`
- le passage, avec son journal et sa date : `f"{passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y} :\n{passages.text[i]}"`
- `retour.choices[0].message.parsed.model_dump()` : les trois champs, en dictionnaire ; `{"passage_id": ..., **champs}` les ajoute à l'identifiant
- `LengthFinishReasonError` (la réponse a atteint `max_tokens`) et `ValidationError` (un JSON illisible, importée de `pydantic`) : comptez la question comme inexploitable
- `executeur.map(generer_question, tirage.index)` : `tirage.index` donne les positions des passages tirés

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
CONSIGNES_GENERATION = """Tu prépares le jeu de test d'un assistant qui répond aux questions sur la presse française de 1914.
À partir de l'extrait d'article fourni, écris une question qu'un lecteur d'aujourd'hui pourrait poser, et sa réponse courte.
- La question se comprend sans l'extrait : elle nomme les personnes, les lieux et les événements dont elle parle ; jamais « le passage », « l'article », « le texte », « ce contexte ».
- Elle porte sur un fait précis que l'extrait donne (qui, quoi, où, combien), et la réponse, en une phrase, s'y trouve.
- Bonnes questions : « Qui a arrêté Villain, l'assassin de Jaurès ? », « Combien de balles ont atteint Gaston Calmette dans l'hôtel du Figaro ? ».
- Mauvaises questions : « Que décrit le passage ? » (pas autonome), « Quel est le rôle du secrétaire dans ce contexte ? » (qui ? quel secrétaire ?).
- Enfin exploitable : false si l'extrait ne donne aucun fait précis (texte trop abîmé, liste, annonce), true sinon."""


def generer_question(i: int) -> dict:
  try:
    retour = client.chat.completions.parse(
      model=MODELE,
      messages=[
        {"role": "system", "content": ...},
        {
          "role": "user",
          "content": f"{passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y} :\n{passages.text[i]}",
        },
      ],
      response_format=...,
      temperature=0,
      max_tokens=200,
      reasoning_effort="none",
    )
  except (LengthFinishReasonError, ValidationError):
    return {
      "passage_id": passages.id[i],
      "question": "",
      "reponse": "",
      "exploitable": False,
    }
  return {"passage_id": passages.id[i], **retour.choices[0].message.parsed.model_dump()}


with ThreadPoolExecutor(8) as executeur:
  nouvelles = list(executeur.map(..., tirage.index))
for generee in nouvelles:
  print(
    f"{generee['passage_id']} (exploitable : {generee['exploitable']})\n"
    f"  Q : {generee['question']}\n  R : {generee['reponse']}\n"
  )
```

### Solution

In [ ]:
CONSIGNES_GENERATION = """Tu prépares le jeu de test d'un assistant qui répond aux questions sur la presse française de 1914.
À partir de l'extrait d'article fourni, écris une question qu'un lecteur d'aujourd'hui pourrait poser, et sa réponse courte.
- La question se comprend sans l'extrait : elle nomme les personnes, les lieux et les événements dont elle parle ; jamais « le passage », « l'article », « le texte », « ce contexte ».
- Elle porte sur un fait précis que l'extrait donne (qui, quoi, où, combien), et la réponse, en une phrase, s'y trouve.
- Bonnes questions : « Qui a arrêté Villain, l'assassin de Jaurès ? », « Combien de balles ont atteint Gaston Calmette dans l'hôtel du Figaro ? ».
- Mauvaises questions : « Que décrit le passage ? » (pas autonome), « Quel est le rôle du secrétaire dans ce contexte ? » (qui ? quel secrétaire ?).
- Enfin exploitable : false si l'extrait ne donne aucun fait précis (texte trop abîmé, liste, annonce), true sinon."""


def generer_question(i: int) -> dict:
  try:
    retour = client.chat.completions.parse(
      model=MODELE,
      messages=[
        {"role": "system", "content": CONSIGNES_GENERATION},
        {
          "role": "user",
          "content": f"{passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y} :\n{passages.text[i]}",
        },
      ],
      response_format=QuestionGeneree,
      temperature=0,
      max_tokens=200,
      reasoning_effort="none",
    )
  except (LengthFinishReasonError, ValidationError):
    # Une réponse coupée par max_tokens, ou un JSON illisible : inexploitable.
    return {
      "passage_id": passages.id[i],
      "question": "",
      "reponse": "",
      "exploitable": False,
    }
  return {"passage_id": passages.id[i], **retour.choices[0].message.parsed.model_dump()}


with ThreadPoolExecutor(8) as executeur:
  nouvelles = list(executeur.map(generer_question, tirage.index))
for generee in nouvelles:
  print(
    f"{generee['passage_id']} (exploitable : {generee['exploitable']})\n"
    f"  Q : {generee['question']}\n  R : {generee['reponse']}\n"
  )

## 2. Filtrer, valider, assembler le jeu mixte

Une question générée n'entre pas telle quelle dans le jeu de test : un filtre automatique écarte d'abord les rebuts évidents, puis un expert relit les autres. `generees` contient 60 questions générées (exercice 1) avec le verdict de l'expert : `valid`, le motif de rejet `reason` (son code, en anglais), une `note`, la réponse de référence (`answer`, corrigée si elle était juste mais mal dite ; la réponse générée reste dans `generated_answer`), `usable` (le champ `exploitable` du modèle) et `passage_ids`, les passages du corpus qui donnent la réponse, le passage source (`source_passage_id`) en premier : l'expert a cherché les autres.

1. Écrivez `motif_filtre(ligne)` : pour une question générée (une ligne de `generees`), le motif de rejet du filtre automatique, ou `None` si elle passe : `"inexploitable"` si le modèle a mis `exploitable` à faux ; `"pas autonome"` si la question renvoie au texte (« ce passage », « le passage », « l'article », « le journal », « ce texte ») ; `"reponse absente"` si moins de la moitié des mots pleins de la réponse générée (ses mots de 4 lettres ou plus) sont dans le passage source.
2. Comparez le filtre et l'expert : combien de questions le filtre écarte-t-il, et combien de celles qu'il garde l'expert rejette-t-il encore ? Comptez les motifs de l'expert, et lisez quelques questions que le filtre a gardées et que l'expert a rejetées.
3. Assemblez `jeu`, le jeu de test : les 20 questions du chapitre 2 (`questions`, `origine` à `"main"`) et les questions générées que l'expert a validées (`origine` à `"generee"`), avec les colonnes `id`, `question`, `answer`, `passage_ids` et `origine`.

**Documentation utile**

- [`re.findall`](https://docs.python.org/3/library/re.html#re.findall)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`DataFrame.assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
- [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

In [ ]:
generees = pd.read_json("synthetic-questions.jsonl", lines=True)

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `set(re.findall(r"\w{4,}", texte.lower()))` : les mots de 4 lettres ou plus d'un texte (`\w` compte les lettres accentuées) ; `len(a & b)` : combien `a` et `b` en ont en commun
- `par_id.text[ligne.source_passage_id]` : le texte du passage source ; `ligne.usable`, `ligne.question`, `ligne.generated_answer` : les champs de la question générée
- `renvoi in ligne.question.lower()` : la question contient-elle `renvoi` ? Le modèle écrit l'apostrophe droite ou typographique : `.replace("’", "'")` les confond
- `generees["filtre"] = [motif_filtre(ligne) for ligne in generees.itertuples()]`, puis `value_counts(dropna=False)` compte aussi les `None` (les questions gardées)
- `pd.crosstab(generees.filtre.fillna("gardee"), generees.valid)` : le filtre en lignes, l'expert en colonnes
- `questions.assign(origine="main")` ajoute la colonne ; `pd.concat([...], ignore_index=True)` met les deux tables bout à bout

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
RENVOIS = ("ce passage", "le passage", "l'article", "le journal", "ce texte")


def mots_pleins(texte: str) -> set[str]:
  return set(re.findall(r"\w{4,}", texte.lower()))


def motif_filtre(ligne) -> str | None:
  if not ligne.usable:
    return "inexploitable"
  if any(renvoi in ligne.question.lower().replace("’", "'") for renvoi in ...):
    return "pas autonome"
  mots = mots_pleins(ligne.generated_answer)
  if len(mots & mots_pleins(par_id.text[...])) < len(mots) / 2:
    return "reponse absente"
  return None


generees["filtre"] = [motif_filtre(ligne) for ligne in generees.itertuples()]
print(generees.filtre.value_counts(dropna=False), "\n")
print(pd.crosstab(generees.filtre.fillna("gardee"), generees.valid), "\n")
print(generees[~generees.valid].reason.value_counts(), "\n")
for ligne in generees[generees.filtre.isna() & ~generees.valid].head(4).itertuples():
  print(
    f"{ligne.question}\n  {ligne.generated_answer}\n  {ligne.reason} : {ligne.note}\n"
  )

jeu = pd.concat(
  [questions.assign(origine=...), generees[generees.valid].assign(origine=...)],
  ignore_index=True,
)[["id", "question", "answer", "passage_ids", "origine"]]
print(jeu.origine.value_counts())
```

### Solution

In [ ]:
RENVOIS = ("ce passage", "le passage", "l'article", "le journal", "ce texte")


def mots_pleins(texte: str) -> set[str]:
  return set(re.findall(r"\w{4,}", texte.lower()))


def motif_filtre(ligne) -> str | None:
  if not ligne.usable:
    return "inexploitable"
  if any(renvoi in ligne.question.lower().replace("’", "'") for renvoi in RENVOIS):
    return "pas autonome"
  mots = mots_pleins(ligne.generated_answer)
  if len(mots & mots_pleins(par_id.text[ligne.source_passage_id])) < len(mots) / 2:
    return "reponse absente"
  return None


generees["filtre"] = [motif_filtre(ligne) for ligne in generees.itertuples()]
print(generees.filtre.value_counts(dropna=False), "\n")
print(pd.crosstab(generees.filtre.fillna("gardee"), generees.valid), "\n")
print(generees[~generees.valid].reason.value_counts(), "\n")
for ligne in generees[generees.filtre.isna() & ~generees.valid].head(4).itertuples():
  print(
    f"{ligne.question}\n  {ligne.generated_answer}\n  {ligne.reason} : {ligne.note}\n"
  )

jeu = pd.concat(
  [
    questions.assign(origine="main"),
    generees[generees.valid].assign(origine="generee"),
  ],
  ignore_index=True,
)[["id", "question", "answer", "passage_ids", "origine"]]
print(jeu.origine.value_counts())

# Le filtre écarte 13 questions, dont 4 que l'expert a validées (deux réponses reformulées,
# une question qui cite « l'article » mais nomme ses personnes, une convocation que le
# modèle jugeait inexploitable). Des 47 qu'il garde, l'expert en rejette encore 8 : des
# opinions, une prémisse fausse, une réponse fausse, des questions qui renvoient au texte
# sans le dire. Le filtre dégrossit ; il ne remplace pas la relecture.

## 3. Mesurer la recherche

La recherche se mesure sans LLM, en quelques secondes par configuration. Écrivez `mesurer(ligne, configuration, k)` : pour une question du jeu qui a une réponse dans le corpus (une ligne de `avec_reponse`), un dictionnaire de trois mesures, d'après le classement de `classer(ligne.question, configuration)` :

- `reussite` : `True` si un passage de référence (`ligne.passage_ids`) est parmi les `k` premiers (le taux de réussite à `k` en est la moyenne) ;
- `precision` : la part des `k` premiers passages qui sont des passages de référence ;
- `rr` : `1 / rang` du premier passage de référence dans le classement, 0 s'il n'y est pas (le MRR en est la moyenne).

Construisez `recherche`, une table d'une ligne par configuration et par question de `evaluees` (la configuration, l'`id`, l'`origine` et les trois mesures, à `k=5`), et affichez les moyennes par configuration et par origine. Que gagne la recherche hybride sur la dense, et le reclassement sur l'hybride ? Les questions générées et celles du chapitre 2 réussissent-elles autant ?

**Documentation utile**

- [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [Agrégation nommée](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation)

In [ ]:
avec_reponse = jeu[jeu.passage_ids.str.len() > 0]
# Les questions évaluées par configuration : toutes ; sans GPU, le reclassement sur 4
# seulement, deux de chaque origine que la recherche dense manque (huit minutes environ,
# durée indicative : le
# cross-encodeur y lit chaque passage en deux à trois secondes).
evaluees = {
  "dense": avec_reponse,
  "hybride": avec_reponse,
  "hybride+reclassement": (
    avec_reponse if GPU else avec_reponse[avec_reponse.id.isin([2, 4, 107, 120])]
  ),
}

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `passages.id[classer(ligne.question, configuration)].tolist()` : les identifiants des passages, dans l'ordre du classement
- `sum(id_passage in pertinents for id_passage in servis) / k` : la précision (`True` compte pour 1)
- `enumerate(classement, start=1)` : chaque identifiant avec son rang, à partir de 1
- `{"configuration": configuration, "id": ligne.id, "origine": ligne.origine, **mesurer(ligne, configuration)}` : une ligne de `recherche`
- `recherche.groupby(["configuration", "origine"]).agg(n=("id", "size"), reussite=("reussite", "mean"), ...)` : une colonne par mesure, avec le nombre de questions
- `memes` : les trois configurations sur les questions du reclassement, pour les comparer sur les mêmes questions
- un exemple à vérifier à la main : un classement dont le premier passage de référence est au rang 2 donne `reussite=True`, `rr=0.5`, et `precision=0.2` si c'est le seul des 5 premiers

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def mesurer(ligne, configuration: str, k: int = 5) -> dict:
  classement = passages.id[classer(ligne.question, configuration)].tolist()
  servis = classement[:k]
  pertinents = set(ligne.passage_ids)
  rangs = [
    rang for rang, id_passage in enumerate(classement, start=1) if id_passage in ...
  ]
  return {
    "reussite": bool(pertinents & set(servis)),
    "precision": sum(id_passage in pertinents for id_passage in servis) / ...,
    "rr": 1 / rangs[0] if rangs else ...,
  }


recherche = pd.DataFrame(
  [
    {
      "configuration": configuration,
      "id": ligne.id,
      "origine": ligne.origine,
      **mesurer(ligne, configuration),
    }
    for configuration, questions_evaluees in evaluees.items()
    for ligne in questions_evaluees.itertuples()
  ]
)
print(
  recherche.groupby(["configuration", "origine"])
  .agg(
    n=("id", "size"),
    reussite=("reussite", "mean"),
    precision=(..., "mean"),
    mrr=(..., "mean"),
  )
  .round(2)
)
memes = recherche[recherche.id.isin(evaluees["hybride+reclassement"].id)]
print(memes.groupby("configuration")[["reussite", "precision", "rr"]].mean().round(2))
```

### Solution

In [ ]:
def mesurer(ligne, configuration: str, k: int = 5) -> dict:
  classement = passages.id[classer(ligne.question, configuration)].tolist()
  servis = classement[:k]
  pertinents = set(ligne.passage_ids)
  rangs = [
    rang
    for rang, id_passage in enumerate(classement, start=1)
    if id_passage in pertinents
  ]
  return {
    "reussite": bool(pertinents & set(servis)),
    "precision": sum(id_passage in pertinents for id_passage in servis) / k,
    "rr": 1 / rangs[0] if rangs else 0.0,
  }


recherche = pd.DataFrame(
  [
    {
      "configuration": configuration,
      "id": ligne.id,
      "origine": ligne.origine,
      **mesurer(ligne, configuration),
    }
    for configuration, questions_evaluees in evaluees.items()
    for ligne in questions_evaluees.itertuples()
  ]
)
print(
  recherche.groupby(["configuration", "origine"])
  .agg(
    n=("id", "size"),
    reussite=("reussite", "mean"),
    precision=("precision", "mean"),
    mrr=("rr", "mean"),
  )
  .round(2)
)
# Les trois configurations sur les mêmes questions, celles du reclassement :
memes = recherche[recherche.id.isin(evaluees["hybride+reclassement"].id)]
print(memes.groupby("configuration")[["reussite", "precision", "rr"]].mean().round(2))

# Lors d'une exécution sans GPU : la recherche hybride retrouve un passage de référence
# dans les 5 premiers pour 89 % des questions du chapitre 2 et 81 % des générées,
# contre 61 % et 63 % pour la dense seule ; sur les 4 questions reclassées, que la dense
# manque, le reclassement les place toutes dans les 5 premiers. Les questions générées ne
# réussissent pas mieux que les autres : leur générateur, un modèle fort, reformule, et
# cite souvent le journal et la date (« d'après L'Humanité du 5 janvier 1914 »), que le
# texte des passages ne contient pas. Le biais connu des questions générées (les mots de
# leur passage) dépend du générateur : on mesure chaque origine à part, sans le supposer.

## 4. Mesurer la réponse avec Ragas

La version « avant » du RAG cherche en dense seule, la version « après » en hybride avec reclassement. `reponses` donne leurs réponses : sur GPU, aux questions du jeu, générées par la cellule ci-dessous ; sans GPU, Qwen3-1.7B y mettrait deux à trois minutes par réponse, et la cellule lit celles que ce même code a produites avec ce même modèle, sur une partie des questions du jeu (`annotated-answers.jsonl`, que l'exercice 5 utilise aussi). Chaque réponse a sa question, sa configuration, `trouve` (`False` pour un refus), `reponse`, `servis` (les 5 passages du prompt) et `sources` (ceux qu'elle cite). Lancez la cellule.

In [ ]:
AVANT, APRES = "dense", "hybride+reclassement"
# Les réponses annotées, avec leurs questions (exercice 5).
annotees = (
  pd.read_json("annotated-answers.jsonl", lines=True)
  .rename(
    columns={
      "id": "numero",
      "question_id": "id",
      "found": "trouve",
      "response": "reponse",
      "served_passage_ids": "servis",
      "cited_passage_ids": "sources",
      "faithful": "fidele_humain",
      "correct": "juste_humain",
    }
  )
  .merge(jeu[["id", "origine", "answer", "passage_ids"]], on="id")
)
colonnes = ["id", "origine", "question", "answer", "passage_ids", "configuration"]
if GPU:
  taches = [
    (ligne, configuration)
    for configuration in (AVANT, APRES)
    for ligne in jeu.itertuples()
  ]
  # Les classements d'abord, un par un : le GPU reclasse une question à la fois.
  for ligne, configuration in taches:
    classer(ligne.question, configuration)
  with ThreadPoolExecutor(8) as executeur:
    resultats = list(
      executeur.map(lambda tache: repondre(tache[0].question, tache[1]), taches)
    )
  reponses = pd.DataFrame(
    [
      {
        **{nom: getattr(ligne, nom) for nom in colonnes[:-1]},
        "configuration": configuration,
        **resultat,
      }
      for (ligne, configuration), resultat in zip(taches, resultats)
    ]
  )
else:
  reponses = annotees[[*colonnes, "trouve", "reponse", "servis", "sources"]]
print(
  reponses.groupby("configuration").trouve.agg(n="size", part_repondue="mean").round(2)
)

### Noter les réponses

Ragas fait juger chaque réponse par un LLM, le juge, servi par le même Ollama. Créez-le avec `llm_factory(MODELE_JUGE, ...)`, puis les deux mesures :

- `Faithfulness`, la fidélité : Ragas découpe la réponse en affirmations, et le juge vérifie chacune contre les passages servis ; le score est la part des affirmations soutenues ;
- `AnswerAccuracy`, l'exactitude : le juge compare la réponse à la référence, puis la référence à la réponse, et note chaque fois 0, 2 ou 4 ; le score est leur moyenne, ramenée entre 0 et 1. Deux appels courts, là où `FactualCorrectness` découpe en affirmations la réponse et la référence avant de les vérifier l'une contre l'autre : plus d'appels, pour des références d'une phrase.

Écrivez `async def noter(ligne)` : pour une réponse (une ligne de `reponses`), un dictionnaire avec `fidelite` et `exactitude`. Un refus n'affirme rien : pas de fidélité (`np.nan`), et une exactitude de 0 si la question a une réponse, sans appeler le juge. Une question sans réponse n'a pas de référence : pas d'exactitude. Le refus et les citations se vérifient par un script : écrivez aussi `verifier(ligne)`, qui renvoie `refus_juste` (le RAG refuse si et seulement si la question est sans réponse) et `citation_juste` (1 si une réponse donnée à une question qui a une réponse cite un passage de référence, 0 sinon ; `np.nan` pour un refus ou une question sans réponse). Vérifiez toutes les réponses de `reponses`, notez celles de `a_noter`, et affichez les moyennes par configuration.

Sans GPU, le juge est le modèle jugé lui-même, Qwen3-1.7B : il tend à approuver ses propres réponses (biais d'auto-préférence), et il est petit. On l'accepte ici, faute d'un modèle plus gros qui tienne la demi-heure sur le processeur de Colab : le protocole est le même, et l'exercice 5 mesure jusqu'où croire ses notes.

**Documentation utile**

- [Ragas : `llm_factory`](https://docs.ragas.io/en/stable/references/llms/#ragas.llms.llm_factory)
- [Ragas : `Faithfulness`](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/)
- [Ragas : `AnswerAccuracy`](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/nvidia_metrics/#answer-accuracy)
- [`asyncio.gather`](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather)

In [ ]:
# Les réponses notées par le juge : toutes ; sans GPU, les 4 réponses de 2 questions, avant
# et après (dix minutes environ, durée indicative : quatre appels au juge par réponse qui
# n'est pas un refus).
a_noter = reponses if GPU else reponses[reponses.id.isin([10, 13])]

# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `llm_factory(MODELE_JUGE, client=AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=3600), mode=instructor.Mode.JSON, temperature=0, max_tokens=1024, reasoning_effort="none")` : le juge, à température 0 ; `instructor.Mode.JSON` lui fait écrire du JSON d'après le schéma placé dans le prompt (`import instructor`)
- `from ragas.metrics.collections import AnswerAccuracy, Faithfulness`, puis `Faithfulness(llm=llm)` et `AnswerAccuracy(llm=llm)`
- `await fidelite.ascore(user_input=question, response=reponse, retrieved_contexts=[...])` et `await exactitude.ascore(user_input=question, response=reponse, reference=answer)` : un résultat dont `.value` est le score. Dans un notebook, `await` s'écrit directement dans la cellule, et `score()`, sa version synchrone, refuse de tourner
- les contextes : chaque passage servi tel que le prompt le montre, avec son journal et sa date (`par_id.loc[id_passage]`) : la réponse peut citer la date d'un passage
- un appel au juge peut échouer (JSON illisible après les nouvelles tentatives) : `try`/`except Exception`, et `np.nan` à la place du score
- `await asyncio.gather(*(noter(ligne) for ligne in a_noter.itertuples()))` : les réponses notées en parallèle, la liste des résultats dans l'ordre
- `pd.DataFrame(liste, index=a_noter.index)` puis `reponses.join(...)` : les mesures à côté des réponses (`np.nan` pour celles qui n'ont pas été notées)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import instructor
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy, Faithfulness

llm = llm_factory(
  MODELE_JUGE,
  client=AsyncOpenAI(
    base_url="http://localhost:11434/v1", api_key="ollama", timeout=3600
  ),
  mode=instructor.Mode.JSON,
  temperature=0,
  max_tokens=1024,
  reasoning_effort="none",
)
fidelite = Faithfulness(llm=...)
exactitude = AnswerAccuracy(llm=...)


def contexte(id_passage: str) -> str:
  passage = par_id.loc[id_passage]
  return f"({passage.newspaper}, {passage.date:%d/%m/%Y}) {passage.text}"


def verifier(ligne) -> dict:
  avec_reponse = ligne.answer is not None
  return {
    "refus_juste": ligne.trouve == ...,
    "citation_juste": (
      float(bool(set(ligne.sources) & set(ligne.passage_ids)))
      if ligne.trouve and avec_reponse
      else np.nan
    ),
  }


async def juger(mesure, **entrees) -> float:
  try:
    return (await mesure.ascore(**entrees)).value
  except Exception as erreur:
    print("Le juge a échoué :", erreur)
    return np.nan


async def noter(ligne) -> dict:
  if not ligne.trouve:
    return {
      "fidelite": np.nan,
      "exactitude": 0.0 if ligne.answer is not None else np.nan,
    }
  notes = {
    "fidelite": await juger(
      ...,
      user_input=ligne.question,
      response=ligne.reponse,
      retrieved_contexts=[contexte(i) for i in ligne.servis],
    ),
    "exactitude": np.nan,
  }
  if ligne.answer is not None:
    notes["exactitude"] = await juger(
      ..., user_input=ligne.question, response=ligne.reponse, reference=...
    )
  return notes


verifications = pd.DataFrame(
  [verifier(ligne) for ligne in reponses.itertuples()], index=reponses.index
)
notes = await asyncio.gather(*(noter(ligne) for ligne in a_noter.itertuples()))
evaluation = reponses.join(verifications).join(pd.DataFrame(notes, index=a_noter.index))
print(
  evaluation.groupby("configuration")[
    ["refus_juste", "citation_juste", "fidelite", "exactitude"]
  ]
  .mean()
  .round(2)
)
```

#### Solution

In [ ]:
import instructor
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy, Faithfulness

llm = llm_factory(
  MODELE_JUGE,
  client=AsyncOpenAI(
    base_url="http://localhost:11434/v1", api_key="ollama", timeout=3600
  ),
  mode=instructor.Mode.JSON,
  temperature=0,
  max_tokens=1024,
  reasoning_effort="none",
)
fidelite = Faithfulness(llm=llm)
exactitude = AnswerAccuracy(llm=llm)


def contexte(id_passage: str) -> str:
  passage = par_id.loc[id_passage]
  return f"({passage.newspaper}, {passage.date:%d/%m/%Y}) {passage.text}"


def verifier(ligne) -> dict:
  avec_reponse = ligne.answer is not None
  return {
    "refus_juste": ligne.trouve == avec_reponse,
    "citation_juste": (
      float(bool(set(ligne.sources) & set(ligne.passage_ids)))
      if ligne.trouve and avec_reponse
      else np.nan
    ),
  }


async def juger(mesure, **entrees) -> float:
  try:
    return (await mesure.ascore(**entrees)).value
  except Exception as erreur:
    print("Le juge a échoué :", erreur)
    return np.nan


async def noter(ligne) -> dict:
  if not ligne.trouve:
    # Un refus n'affirme rien, et ne donne pas la réponse attendue : pas de juge.
    return {
      "fidelite": np.nan,
      "exactitude": 0.0 if ligne.answer is not None else np.nan,
    }
  notes = {
    "fidelite": await juger(
      fidelite,
      user_input=ligne.question,
      response=ligne.reponse,
      retrieved_contexts=[contexte(i) for i in ligne.servis],
    ),
    "exactitude": np.nan,
  }
  if ligne.answer is not None:
    notes["exactitude"] = await juger(
      exactitude,
      user_input=ligne.question,
      response=ligne.reponse,
      reference=ligne.answer,
    )
  return notes


verifications = pd.DataFrame(
  [verifier(ligne) for ligne in reponses.itertuples()], index=reponses.index
)
notes = await asyncio.gather(*(noter(ligne) for ligne in a_noter.itertuples()))
evaluation = reponses.join(verifications).join(pd.DataFrame(notes, index=a_noter.index))
print(
  evaluation.groupby("configuration")[
    ["refus_juste", "citation_juste", "fidelite", "exactitude"]
  ]
  .mean()
  .round(2)
)

# Lors d'une exécution sans GPU : avant, les deux réponses notées sont des refus (la
# recherche dense n'a pas servi le passage), d'où une exactitude de 0 ; après, toutes deux
# sont justes. Leur fidélité moyenne, 0,5, vient du juge : l'annotation de l'exercice 5
# les tient toutes deux pour fidèles.

## 5. Contrôler le juge

Le juge se trompe aussi : avant de croire ses notes, on mesure son accord avec un annotateur, l'expert de l'exercice 2. `annotees` (lue à l'exercice 4) contient des réponses réelles du RAG (Qwen3-1.7B, avant et après), chacune annotée : `fidele_humain` (la réponse ne dit rien que les passages servis ne disent), `juste_humain` (elle dit ce que dit la référence ; un refus est juste si la question est sans réponse) et une `note`. Notez celles de `controlees` (les réponses qui ne sont pas des refus ; sans GPU, 3) avec `noter`. Le juge dit « fidèle » si `fidelite` vaut au moins 0,5, « juste » si `exactitude` vaut au moins 0,5. Calculez, critère par critère, la part des réponses où il est d'accord avec l'humain, sur celles qu'il a notées, et lisez les désaccords : que manque-t-il au juge ?

In [ ]:
print(f"{len(annotees)} réponses annotées :")
print(pd.crosstab(annotees.fidele_humain, annotees.juste_humain), "\n")
# Les réponses contrôlées : celles qui ne sont pas des refus (les refus se vérifient par
# script) ; sans GPU, 3 d'entre elles (un quart d'heure environ, durée indicative : quatre
# appels au juge par réponse).
non_refus = annotees[annotees.trouve]
controlees = non_refus if GPU else non_refus[non_refus.numero.isin([5, 9, 33])]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `await asyncio.gather(*(noter(ligne) for ligne in controlees.itertuples()))`, puis `controlees.join(pd.DataFrame(notes, index=controlees.index))`, comme à l'exercice 4
- pour chaque critère, `(critere, mesure)` dans `(("fidele", "fidelite"), ("juste", "exactitude"))` : les réponses notées sont celles où la mesure n'est pas `NaN` (`notna()`), le verdict du juge est `mesure >= 0.5`, celui de l'humain la colonne `f"{critere}_humain"`
- `(juge == humain).mean()` : la part d'accord

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
notes = await asyncio.gather(*(noter(ligne) for ligne in controlees.itertuples()))
controle = controlees.join(pd.DataFrame(notes, index=controlees.index))
for critere, mesure in (("fidele", "fidelite"), ("juste", "exactitude")):
  notees = controle[controle[mesure].notna()]
  juge = notees[mesure] >= ...
  humain = notees[f"{critere}_humain"]
  print(
    f"{critere} : le juge est d'accord avec l'humain sur {(juge == humain).mean():.0%} de {len(notees)} réponses"
  )
  for ligne in notees[juge != ...].itertuples():
    print(
      f"  Q{ligne.id} ({ligne.configuration}) {ligne.question}\n    réponse : {ligne.reponse}\n"
      f"    juge {getattr(ligne, mesure):.2f}, humain {getattr(ligne, f'{critere}_humain')} : {ligne.note}"
    )
```

### Solution

In [ ]:
notes = await asyncio.gather(*(noter(ligne) for ligne in controlees.itertuples()))
controle = controlees.join(pd.DataFrame(notes, index=controlees.index))
for critere, mesure in (("fidele", "fidelite"), ("juste", "exactitude")):
  notees = controle[controle[mesure].notna()]
  juge = notees[mesure] >= 0.5
  humain = notees[f"{critere}_humain"]
  print(
    f"{critere} : le juge est d'accord avec l'humain sur {(juge == humain).mean():.0%} "
    f"de {len(notees)} réponses"
  )
  for ligne in notees[juge != humain].itertuples():
    print(
      f"  Q{ligne.id} ({ligne.configuration}) {ligne.question}\n"
      f"    réponse : {ligne.reponse}\n"
      f"    juge {getattr(ligne, mesure):.2f}, humain {getattr(ligne, f'{critere}_humain')} : "
      f"{ligne.note}"
    )

# Lors d'une exécution sans GPU, le juge Qwen3-1.7B se montre indulgent : il tient pour
# fidèle « cinq balles », un chiffre qu'aucun passage servi ne donne, juste à moitié
# (0,5) cette même réponse fausse, et juste une réponse qui ne fait que reformuler la
# question. Un juge plus fort, d'une autre famille (sur GPU), ou une grille plus stricte
# (« juste » seulement à 1) se contrôlent de la même façon, sur le même échantillon.

## 6. Consigner et comparer

Une évaluation ne vaut que comparée à une autre, sur le même jeu. Écrivez `consigner(configuration, mesures)` : elle ajoute à `evaluations.jsonl` une ligne JSON avec la date, l'empreinte du jeu (`EMPREINTE`, ci-dessous : une question changée la change), la configuration, les modèles et les mesures. Consignez la recherche (une ligne par configuration de `recherche`) et les réponses (une par configuration de `evaluation`), puis relisez le fichier.

Écrivez ensuite `comparer(avant, apres)` : deux séries de succès par question, indexées par `id`, elle renvoie les questions gagnées (échec avant, succès après) et les perdues, sur les questions des deux. Comparez la recherche dense et l'hybride avec reclassement (`reussite`), puis les réponses avant et après (juste si `exactitude` vaut au moins 0,5, ou si le RAG a refusé à raison une question sans réponse). Sur `n` questions, un taux `p` a une marge d'environ ±1,96·√(p(1−p)/n) : la différence entre avant et après la dépasse-t-elle ?

**Documentation utile**

- [`json.dumps`](https://docs.python.org/3/library/json.html#json.dumps)
- [`datetime.now`](https://docs.python.org/3/library/datetime.html#datetime.datetime.now)
- [`DataFrame.pivot`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)

In [ ]:
# L'empreinte du jeu : les 12 premiers caractères du SHA-256 de son contenu.
EMPREINTE = hashlib.sha256(
  jeu.to_json(orient="records", force_ascii=False).encode()
).hexdigest()[:12]
MODELES = {
  "encodeur": ENCODEUR,
  "reclasseur": RECLASSEUR,
  "llm": MODELE,
  "juge": MODELE_JUGE,
}


def marge(p: float, n: int) -> float:
  return 1.96 * np.sqrt(p * (1 - p) / n)


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `datetime.now().isoformat(timespec="seconds")` : la date ; `json.dumps(ligne, ensure_ascii=False)` : la ligne, gardant les accents ; `open("evaluations.jsonl", "a")` ajoute à la fin du fichier
- `float(...)` et `int(...)` autour des moyennes : `json` refuse les nombres de NumPy
- `for configuration, groupe in recherche.groupby("configuration")` : les questions d'une configuration
- `recherche.pivot(index="id", columns="configuration", values="reussite")` : une colonne de succès par configuration ; `.dropna()` garde les questions évaluées
- le succès d'une réponse : `(evaluation.exactitude >= 0.5) | (evaluation.answer.isna() & evaluation.refus_juste)`, sur les réponses notées (`a_noter.index`)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def consigner(configuration: dict, mesures: dict) -> None:
  ligne = {
    "date": datetime.now().isoformat(timespec="seconds"),
    "jeu": EMPREINTE,
    "configuration": configuration,
    "modeles": MODELES,
    "mesures": ...,
  }
  with open("evaluations.jsonl", "a") as fichier:
    fichier.write(json.dumps(ligne, ensure_ascii=False) + "\n")


for configuration, groupe in recherche.groupby("configuration"):
  consigner(
    {"etape": "recherche", "recherche": configuration, "k": 5},
    {
      "n": len(groupe),
      "reussite": float(groupe.reussite.mean()),
      "mrr": float(groupe.rr.mean()),
    },
  )
notees = evaluation.loc[a_noter.index]
for configuration, groupe in notees.groupby("configuration"):
  consigner(
    {"etape": "reponse", "recherche": configuration, "k": 5},
    {
      "n": len(groupe),
      "fidelite": float(groupe.fidelite.mean()),
      "exactitude": float(groupe.exactitude.mean()),
    },
  )
journal = pd.json_normalize([json.loads(ligne) for ligne in open("evaluations.jsonl")])
print(
  journal.drop(columns=journal.filter(like="modeles").columns).round(2).to_string(),
  "\n",
)


def comparer(avant: pd.Series, apres: pd.Series) -> tuple[list, list]:
  communes = avant.index.intersection(apres.index)
  gagnees = [i for i in communes if not avant[i] and apres[i]]
  perdues = [i for i in communes if ... and not ...]
  return gagnees, perdues


for titre, succes in (
  (
    "recherche",
    recherche.pivot(index="id", columns="configuration", values="reussite"),
  ),
  (
    "réponse",
    notees.assign(
      succes=(notees.exactitude >= 0.5) | (notees.answer.isna() & notees.refus_juste)
    ).pivot(index="id", columns="configuration", values="succes"),
  ),
):
  communes = succes[[AVANT, APRES]].dropna().astype(bool)
  gagnees, perdues = comparer(communes[AVANT], communes[APRES])
  n = len(communes)
  for nom in (AVANT, APRES):
    p = communes[nom].mean()
    print(f"{titre}, {nom} : {p:.0%} ± {marge(p, n):.0%} sur {n} questions")
  print(f"  gagnées : {gagnees}, perdues : {perdues}\n")
```

### Solution

In [ ]:
def consigner(configuration: dict, mesures: dict) -> None:
  ligne = {
    "date": datetime.now().isoformat(timespec="seconds"),
    "jeu": EMPREINTE,
    "configuration": configuration,
    "modeles": MODELES,
    "mesures": mesures,
  }
  with open("evaluations.jsonl", "a") as fichier:
    fichier.write(json.dumps(ligne, ensure_ascii=False) + "\n")


for configuration, groupe in recherche.groupby("configuration"):
  consigner(
    {"etape": "recherche", "recherche": configuration, "k": 5},
    {
      "n": len(groupe),
      "reussite": float(groupe.reussite.mean()),
      "precision": float(groupe.precision.mean()),
      "mrr": float(groupe.rr.mean()),
    },
  )
notees = evaluation.loc[a_noter.index]
for configuration, groupe in notees.groupby("configuration"):
  consigner(
    {"etape": "reponse", "recherche": configuration, "k": 5, "prompt": "INSTRUCTIONS"},
    {
      "n": len(groupe),
      "refus_juste": float(groupe.refus_juste.mean()),
      "fidelite": float(groupe.fidelite.mean()),
      "exactitude": float(groupe.exactitude.mean()),
    },
  )
journal = pd.json_normalize([json.loads(ligne) for ligne in open("evaluations.jsonl")])
print(
  journal.drop(columns=journal.filter(like="modeles").columns).round(2).to_string(),
  "\n",
)


def comparer(avant: pd.Series, apres: pd.Series) -> tuple[list, list]:
  communes = avant.index.intersection(apres.index)
  gagnees = [i for i in communes if not avant[i] and apres[i]]
  perdues = [i for i in communes if avant[i] and not apres[i]]
  return gagnees, perdues


succes_reponse = (notees.exactitude >= 0.5) | (
  notees.answer.isna() & notees.refus_juste
)
for titre, succes in (
  (
    "recherche",
    recherche.pivot(index="id", columns="configuration", values="reussite"),
  ),
  (
    "réponse",
    notees.assign(succes=succes_reponse).pivot(
      index="id", columns="configuration", values="succes"
    ),
  ),
):
  communes = succes[[AVANT, APRES]].dropna().astype(bool)
  gagnees, perdues = comparer(communes[AVANT], communes[APRES])
  n = len(communes)
  for nom in (AVANT, APRES):
    p = communes[nom].mean()
    print(f"{titre}, {nom} : {p:.0%} ± {marge(p, n):.0%} sur {n} questions")
  print(f"  gagnées : {gagnees}, perdues : {perdues}\n")

# Sans GPU, la comparaison porte sur 4 questions pour la recherche et 2 pour la réponse :
# des gains sans valeur statistique, et une marge nulle quand le taux vaut 0 ou 1 (la
# formule, une approximation, n'y vaut plus). C'est sur le jeu entier que la comparaison
# avant/après compte.